In [ ]:
# %% [markdown]
# ## Cell 1 — Clone + Install (MINIMAL — no imports, no numpy ABI risk)

# %%
import os
import subprocess
import sys
import glob

REPO_URL = "https://github.com/romin4444/multimodal-financial-crisis-prediction.git"
REPO_DIR = "/kaggle/working/fcps"

print("\n" + "="*70)
print("CELL 1: Clone repo + install FCPS")
print("="*70)

# Step 1: Clone or pull
if os.path.isdir(REPO_DIR):
    print(f"[1] Repo exists — pulling...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], 
                   capture_output=True, text=True)
else:
    print(f"[1] Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

# Step 2: Get current HEAD
head = subprocess.run(["git", "log", "-1", "--oneline"], 
                      capture_output=True, text=True).stdout.strip()
print(f"    HEAD: {head}")

# Step 3: Install FCPS core
print("[2] Installing FCPS core (--no-deps)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], 
               check=True)
print("    ✓ Done")

# Step 4: Try to install FinBERT (but don't fail if it doesn't work)
print("[3] Installing FinBERT (optional)...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
                   "transformers>=4.30", "datasets>=2.14", "--no-deps"],
                   capture_output=True)
if r.returncode == 0:
    print("    ✓ FinBERT installed")
else:
    print("    ~ FinBERT skipped (scripts will still run)")

# Step 5: List scripts
scripts = sorted(os.path.basename(p) for p in glob.glob("scripts/*.py"))
print(f"\n[OK] {len(scripts)} scripts ready.")
print("    Proceed to Cell 2.\n")

In [ ]:
# %% [markdown]
# ## Cell 1.5 — Apply v3.4 hazard calibration fix (overlay onto the fresh clone)
#
# The v3.3 hazard calibrator fit isotonic regression on the TEMPORAL TAIL of the
# train window (~2008-2010 GFC), so it over-predicted on the calmer test set and
# turned a healthy raw Brier skill NEGATIVE. v3.4 replaces it with a low-variance
# Platt/sigmoid fit on time-series OUT-OF-FOLD predictions + a do-no-harm guard
# (falls back to raw when calibration can't be confirmed to help).
#
# This cell overlays the patched files so you can run + verify on Kaggle BEFORE
# committing them to GitHub. Once happy: commit src/v3/hazard.py, scripts/hazard_run.py
# (+ the new tests) and this cell becomes a no-op against your updated remote.

import base64, pathlib

_HAZARD_B64 = "..."
_HAZARD_RUN_B64 = "IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiIKdjMg4oCUIERpc2NyZXRlLXRpbWUgSEFaQVJEIC8gc3Vydml2YWwgbW9kZWwgcnVuLgoKUmVmcmFtZXMgY3Jpc2lzIG9uc2V0IGFzIHRpbWUtdG8tZXZlbnQ6IGVzdGltYXRlcyB0aGUgcGVyLWRheSBoYXphcmQgb2YgZW50ZXJpbmcKYSA+PTEwJSBkcmF3ZG93biBhbmQgY29udmVydHMgaXQgdG8gUCg+PTEwJSBkcmF3ZG93biB3aXRoaW4gTiBkYXlzKS4gUmVwb3J0cwpjb25jb3JkYW5jZSAoQy1pbmRleCkgYW5kIE4tZGF5IGN1bXVsYXRpdmUtaW5jaWRlbmNlIGNhbGlicmF0aW9uIG9uIGEgaGVsZC1vdXQKdGVtcG9yYWwgdGVzdCBzcGxpdC4KCkF1dGhvcjogUm9taW4gUGF0ZWwuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmlmIHN5cy5wbGF0Zm9ybSA9PSAid2luMzIiOgogICAgc3lzLnN0ZG91dCA9IGlvLlRleHRJT1dyYXBwZXIoc3lzLnN0ZG91dC5idWZmZXIsIGVuY29kaW5nPSJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpCiAgICBzeXMuc3RkZXJyID0gaW8uVGV4dElPV3JhcHBlcihzeXMuc3RkZXJyLmJ1ZmZlciwgZW5jb2Rpbmc9InV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucGFyZW50LnBhcmVudCkpCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKRERfVEhSRVNIT0xEID0gMC4xMApIT1JJWk9OUyA9IFsyMSwgNjNdCgoKZGVmIG1haW4oKSAtPiBkaWN0OgogICAgcHJpbnQoIj0iICogNzIpCiAgICBwcmludCgiICBGQ1BTIHYzIOKAlCBESVNDUkVURS1USU1FIEhBWkFSRCAvIFNVUlZJVkFMIE1PREVMIikKICAgIHByaW50KCI9IiAqIDcyKQoKICAgIGZyb20gc3JjLmNvbmZpZyBpbXBvcnQgY2ZnCiAgICBmcm9tIHNyYy5sb2dnaW5nX3NldHVwIGltcG9ydCBzZXR1cF9sb2dnaW5nCiAgICBzZXR1cF9sb2dnaW5nKGxldmVsPSJXQVJOSU5HIiwgZm10PSJ0ZXh0IikKCiAgICBmcm9tIHNyYy5kYXRhLm1hcmtldCBpbXBvcnQgZG93bmxvYWRfYWxsX21hcmtldAogICAgZnJvbSBzcmMuZGF0YS5mcmVkIGltcG9ydCBkb3dubG9hZF9mcmVkLCBhbGlnbl9mcmVkX3RvX3RyYWRpbmdfZGF5cwogICAgZnJvbSBzcmMuZmVhdHVyZXMuZW5naW5lZXJpbmcgaW1wb3J0IGVuZ2luZWVyX2ZlYXR1cmVzCiAgICBmcm9tIHNyYy5tb2RlbHMuZnNpIGltcG9ydCBGU0lCdWlsZGVyCiAgICBmcm9tIHNyYy52My5tYWNyb19mZWF0dXJlcyBpbXBvcnQgYnVpbGRfbWFjcm9fZmVhdHVyZXMsIHVzYWJsZV9tYWNyb19jb2xzCiAgICBmcm9tIHNyYy52My5oYXphcmQgaW1wb3J0IGRyYXdkb3duX3BhbmVsLCBmaXRfaGF6YXJkLCBldmFsdWF0ZV9oYXphcmQKCiAgICBwcmludCgiXG5bMV0gRGF0YSArIGZlYXR1cmVzLi4uIikKICAgIG1hcmtldCA9IGRvd25sb2FkX2FsbF9tYXJrZXQoKQogICAgZmVhdCA9IGVuZ2luZWVyX2ZlYXR1cmVzKG1hcmtldFsic3A1MDAiXSwgbWFya2V0WyJ2aXgiXSkKICAgIG4gPSBsZW4oZmVhdCkKICAgIHRyYWluX21hc2sgPSBucC56ZXJvcyhuLCBkdHlwZT1ib29sKQogICAgdHJhaW5fbWFza1s6IGludChuICogMC42KV0gPSBUcnVlCiAgICBmZWF0LCBfID0gRlNJQnVpbGRlcigpLmJ1aWxkKGZlYXQsIHBkLkRhdGFGcmFtZSgpLCB0cmFpbl9tYXNrPXRyYWluX21hc2spCiAgICBmcmVkX2RhaWx5ID0gYWxpZ25fZnJlZF90b190cmFkaW5nX2RheXMoZG93bmxvYWRfZnJlZCgpLCBmZWF0LmluZGV4KQogICAgbWFjcm8gPSBidWlsZF9tYWNyb19mZWF0dXJlcyhmcmVkX2RhaWx5LCBtYXJrZXQsIGZlYXQuaW5kZXgpCiAgICBmZWF0ID0gZmVhdC5qb2luKG1hY3JvLCBob3c9ImxlZnQiKQoKICAgIHByaW50KCJbMl0gQnVpbGRpbmcgc3Vydml2YWwgcGFuZWwgKGRyYXdkb3duIG9uc2V0cyAvIGF0LXJpc2sgLyBkdXJhdGlvbikuLi4iKQogICAgcGFuZWwgPSBkcmF3ZG93bl9wYW5lbChmZWF0WyJjbG9zZSJdLCB0aHJlc2hvbGQ9RERfVEhSRVNIT0xEKQogICAgbl9vbnNldHMgPSBpbnQocGFuZWxbIm9uc2V0Il0uc3VtKCkpCiAgICBwcmludChmIiAgICA+PTEwJSBkcmF3ZG93biBlcGlzb2Rlczoge25fb25zZXRzfSAgfCBhdC1yaXNrIGRheXM6IHtpbnQocGFuZWxbJ2F0X3Jpc2snXS5zdW0oKSk6LH0iKQoKICAgICMgRmVhdHVyZXMgZm9yIGhhemFyZDogcHJpY2Ugc3RyZXNzICsgd2VsbC1wb3B1bGF0ZWQgVklYLW9ydGhvZ29uYWwgbWFjcm8KICAgIGZlYXRfY29scyA9IFtjIGZvciBjIGluIFsidm9sXzIxZCIsICJ2aXgiLCAiZHJhd2Rvd25fNjMiLCAibW9tXzIxZCJdIGlmIGMgaW4gZmVhdC5jb2x1bW5zXQogICAgZmVhdF9jb2xzICs9IHVzYWJsZV9tYWNyb19jb2xzKGZlYXQpCgogICAgIyBUZW1wb3JhbCBzcGxpdDogdHJhaW4gb24gZmlyc3QgNjAlLCB0ZXN0IG9uIGxhc3QgNDAlCiAgICBpZHggPSBmZWF0LmluZGV4CiAgICBjdXQgPSBpbnQobGVuKGlkeCkgKiAwLjYpCiAgICB0cl9tYXNrID0gbnAuemVyb3MobGVuKGlkeCksIGR0eXBlPWJvb2wpCiAgICB0cl9tYXNrWzpjdXRdID0gVHJ1ZQogICAgdGVfbWFzayA9IH50cl9tYXNrCiAgICBwcmludChmIlszXSBUcmFpbiB7aWR4WzBdLmRhdGUoKX0tPntpZHhbY3V0LTFdLmRhdGUoKX0gfCBUZXN0IHtpZHhbY3V0XS5kYXRlKCl9LT57aWR4Wy0xXS5kYXRlKCl9IikKCiAgICBwcmludCgiXG5bNF0gT3V0LW9mLXNhbXBsZSBoYXphcmQgZXZhbHVhdGlvbjpcbiIpCiAgICAjIHYzLjQ6IHJlcG9ydCBCT1RIIHRoZSByYXcgMS0oMS1oKV5OIHJpc2sgQU5EIHRoZSBjYWxpYnJhdGVkIHZlcnNpb24sIHBsdXMKICAgICMgd2hpY2ggY2FsaWJyYXRvciBzdXJ2aXZlZCB0aGUgZG8tbm8taGFybSBndWFyZCAoImlkZW50aXR5IiA9IHJhdyB3YXMgYmVzdCkuCiAgICBwcmludChmIiAgeydIb3Jpem9uJzo+OH0geydDLWluZGV4Jzo+OX0geydCYXNlUmF0ZSc6Pjl9IHsnQlNTX3Jhdyc6Pjl9IHsnQlNTX2NhbCc6Pjl9IHsnTWV0aG9kJzo+MTR9IikKICAgIHByaW50KCIgICIgKyAiLSIgKiA3MikKICAgIHJlc3VsdHMgPSB7fQogICAgZmVhdHVyZV9jb2xzX3VzZWQ6IGxpc3Rbc3RyXSA9IFtdCiAgICBmb3IgaCBpbiBIT1JJWk9OUzoKICAgICAgICAjIFRoZSBwZXItc3RlcCBoYXphcmQgbW9kZWwgaXMgaG9yaXpvbi1pbmRlcGVuZGVudCwgYnV0IHRoZSBpc290b25pYwogICAgICAgICMgY2FsaWJyYXRvciBJUyBob3Jpem9uLXNwZWNpZmljIChpdCBtYXBzIDEtKDEtaCleTiB0byByZWFsaXplZCBOLWRheQogICAgICAgICMgZnJlcXVlbmN5KS4gU28gd2UgcmVmaXQgcGVyIGhvcml6b24g4oCUIG1vZGVsIGZpdCBpcyBjaGVhcC4KICAgICAgICBmaXQgPSBmaXRfaGF6YXJkKAogICAgICAgICAgICBwYW5lbCwgZmVhdCwgZmVhdF9jb2xzLCB0cl9tYXNrLAogICAgICAgICAgICBob3Jpem9uPWgsIGNhbGlicmF0ZT1UcnVlLCBkcmF3ZG93bl90aHJlc2hvbGQ9RERfVEhSRVNIT0xELAogICAgICAgICkKICAgICAgICBmZWF0dXJlX2NvbHNfdXNlZCA9IGZpdC5mZWF0dXJlX2NvbHMKICAgICAgICBtID0gZXZhbHVhdGVfaGF6YXJkKAogICAgICAgICAgICBmaXQsIHBhbmVsLCBmZWF0LCBob3Jpem9uPWgsIHRlc3RfbWFzaz10ZV9tYXNrLAogICAgICAgICAgICBkcmF3ZG93bl90aHJlc2hvbGQ9RERfVEhSRVNIT0xELAogICAgICAgICkKICAgICAgICByZXN1bHRzW2YiaHtofSJdID0gbQogICAgICAgIHByaW50KGYiICB7aDo+OH0ge3N0cihtLmdldCgnY19pbmRleCcpKTo+OX0gIgogICAgICAgICAgICAgIGYie3N0cihtLmdldCgnTmRheV9iYXNlX3JhdGUnKSk6Pjl9ICIKICAgICAgICAgICAgICBmIntzdHIobS5nZXQoJ05kYXlfcmlza19icmllcl9za2lsbF9yYXcnKSk6Pjl9ICIKICAgICAgICAgICAgICBmIntzdHIobS5nZXQoJ05kYXlfcmlza19icmllcl9za2lsbF9jYWxpYnJhdGVkJykpOj45fSAiCiAgICAgICAgICAgICAgZiJ7c3RyKG0uZ2V0KCdjYWxpYl9tZXRob2QnKSk6PjE0fSIpCgogICAgZnJvbSBzcmMuanNvbl91dGlscyBpbXBvcnQgc2FmZV9qc29uX2RlZmF1bHQKICAgIG91dCA9IHsKICAgICAgICAidGFzayI6ICJkaXNjcmV0ZS10aW1lIGhhemFyZDogUCg+PTEwJSBkcmF3ZG93biB3aXRoaW4gTiBkYXlzKSIsCiAgICAgICAgImRyYXdkb3duX3RocmVzaG9sZCI6IEREX1RIUkVTSE9MRCwKICAgICAgICAibl9vbnNldHMiOiBuX29uc2V0cywKICAgICAgICAiZmVhdHVyZV9jb2xzIjogZmVhdHVyZV9jb2xzX3VzZWQsCiAgICAgICAgImNhbGlicmF0aW9uIjogInYzLjQ6IFBsYXR0L3NpZ21vaWQgb24gdGltZS1zZXJpZXMgT09GIGluY2lkZW5jZSArIGRvLW5vLWhhcm0gZ3VhcmQgKGZhbGxzIGJhY2sgdG8gcmF3KSIsCiAgICAgICAgInJlc3VsdHMiOiByZXN1bHRzLAogICAgfQogICAgd2l0aCBvcGVuKGNmZy5wYXRocy5vdXRwdXRfZGlyIC8gImhhemFyZF9tZXRyaWNzLmpzb24iLCAidyIpIGFzIGZoOgogICAgICAgIGpzb24uZHVtcChvdXQsIGZoLCBpbmRlbnQ9MiwgZGVmYXVsdD1zYWZlX2pzb25fZGVmYXVsdCkKCiAgICBwcmludCgiXG4iICsgIj0iICogNzIpCiAgICBwcmludCgiICBWRVJESUNUIikKICAgIHByaW50KCI9IiAqIDcyKQogICAgYzIxID0gcmVzdWx0cy5nZXQoImgyMSIsIHt9KS5nZXQoImNfaW5kZXgiKQogICAgcHJpbnQoZiIgIEhhemFyZCBjb25jb3JkYW5jZSAoQy1pbmRleCwgMjFkKToge2MyMX0gICIKICAgICAgICAgIGYiKHsnYmV0dGVyIHRoYW4gY2hhbmNlJyBpZiBpc2luc3RhbmNlKGMyMSwgZmxvYXQpIGFuZCBjMjEgPiAwLjU1IGVsc2UgJ25lYXIgY2hhbmNlJ30pIikKICAgIHByaW50KGYiICBTYXZlZDoge2NmZy5wYXRocy5vdXRwdXRfZGlyIC8gJ2hhemFyZF9tZXRyaWNzLmpzb24nfSIpCiAgICByZXR1cm4gb3V0CgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="

pathlib.Path("/kaggle/working/fcps/src/v3/hazard.py").write_bytes(base64.b64decode(_HAZARD_B64))
pathlib.Path("/kaggle/working/fcps/scripts/hazard_run.py").write_bytes(base64.b64decode(_HAZARD_RUN_B64))
print("[v3.4] hazard calibration fix applied: sigmoid-OOF + do-no-harm guard")
print("       -> hazard_metrics.json will now report a `calib_method` per horizon")


In [ ]:
# ── Secrets + inputs → environment (your fred.py / news.py read these) ────────
import os, glob

# FRED key from Kaggle Secrets → env (fred.py loads os.environ["FRED_API_KEY"])
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_SECRET_FRED_API_KEY"] = UserSecretsClient().get_secret("KAGGLE_SECRET_FRED_API_KEY").strip()
    print("[OK] FRED_API_KEY loaded")
except Exception as e:
    print("[!] No FRED_API_KEY secret — credit/yield/funding + vintage-FSI will be limited:", str(e)[:80])

# Point NEWS_DATA_DIR at the first attached news dataset (load_news auto-detects columns)
news_dirs = sorted(glob.glob("/kaggle/input/datasets")) or sorted(glob.glob("/kaggle/input/datasets"))
if news_dirs:
    os.environ["NEWS_DATA_DIR"] = news_dirs[0]
    print("[OK] NEWS_DATA_DIR =", news_dirs[0])
else:
    print("[!] No news dataset attached — FinBERT ablation will skip cleanly")

# IMPORTANT: the repo ships a 2023-24-only fred_data.csv cache. Delete it so your
# vintage/ALFRED fetch pulls full 1990+ history instead of reusing the short cache.
for c in ["fred_data.csv", "data/fred_data.csv", "data/cache/fred_data.csv"]:
    if os.path.exists(c):
        os.remove(c); print("[removed stale cache]", c)

In [ ]:
import os, subprocess, sys, glob, shutil

os.chdir("/kaggle/working/fcps")
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

SCRIPTS = [
    ("v3 crisis",    "scripts/v3_run.py"),
    ("hazard",       "scripts/hazard_run.py"),
    ("direction",    "scripts/direction_run.py"),
    ("overlay",      "scripts/risk_overlay_run.py"),
    ("v3_advanced",  "scripts/v3_advanced_run.py"),
    ("real_data",    "scripts/real_data_run.py"),
]

for label, script in SCRIPTS:
    if not os.path.exists(script):
        print(f"[SKIP] {script}"); continue
    print(f"\n{'='*60}\n[RUN] {label}\n{'='*60}", flush=True)
    r = subprocess.run([sys.executable, script], capture_output=True, text=True, timeout=1800)
    print((r.stdout + r.stderr)[-3000:])
    print("✓" if r.returncode == 0 else f"✗ code {r.returncode}")

for f in glob.glob("outputs/*.json") + glob.glob("outputs/*.png"):
    shutil.copy(f, "/kaggle/working/artifacts/")

print(f"\n[DONE] {len(os.listdir('/kaggle/working/artifacts'))} artifacts collected")

In [ ]:
# ── Read back the key metrics ────────────────────────────────────────────────
import json, glob, os

for name in ["v3_metrics.json", "hazard_metrics.json", "v3_advanced_metrics.json",
             "risk_overlay_results.json", "direction_metrics.json", "metrics_summary.json"]:
    p = os.path.join("outputs", name)
    if os.path.exists(p):
        print(f"\n===== {name} =====")
        print(json.dumps(json.load(open(p)), indent=2)[:2500])

# Sanity check on the calibration fix: hazard Brier skill should no longer be ~ -2
try:
    hz = json.load(open("outputs/hazard_metrics.json"))
    print("\n>>> hazard C-index & Brier skill:", hz)
except Exception:
    pass

In [ ]:
# %% [markdown]
# ## Cell 5 — REAL-DATA before/after: v3.3 (isotonic_tail) vs v3.4 (sigmoid_oof)
# Same S&P500+VIX+macro features the hazard script uses. Watch the BSS_cal column:
# isotonic_tail drives the 21d calibrated skill negative; sigmoid_oof keeps it >= raw.

import os, sys, numpy as np, pandas as pd
os.chdir("/kaggle/working/fcps"); sys.path.insert(0, "/kaggle/working/fcps")
from src.data.market import download_all_market
from src.features.engineering import engineer_features
from src.data.fred import download_fred, align_fred_to_trading_days
from src.models.fsi import FSIBuilder
from src.v3.macro_features import build_macro_features, usable_macro_cols
from src.v3.hazard import drawdown_panel, fit_hazard, evaluate_hazard

market = download_all_market()
feat = engineer_features(market["sp500"], market["vix"])
n = len(feat); tr = np.zeros(n, dtype=bool); tr[: int(n * 0.6)] = True
feat, _ = FSIBuilder().build(feat, pd.DataFrame(), train_mask=tr)
fred_daily = align_fred_to_trading_days(download_fred(), feat.index)
feat = feat.join(build_macro_features(fred_daily, market, feat.index), how="left")
panel = drawdown_panel(feat["close"], threshold=0.10)
cols = [c for c in ["vol_21d", "vix", "drawdown_63", "mom_21d"] if c in feat.columns] + usable_macro_cols(feat)
te = ~tr

print(f"{'calib_method':>18} {'H':>3} {'C-idx':>7} {'BSS_raw':>9} {'BSS_cal':>9}")
print("-" * 52)
for H in (21, 63):
    for meth in ("isotonic_tail", "sigmoid_oof"):
        fit = fit_hazard(panel, feat, cols, tr, horizon=H, calibrate=True,
                         drawdown_threshold=0.10, calib_method=meth)
        m = evaluate_hazard(fit, panel, feat, horizon=H, test_mask=te, drawdown_threshold=0.10)
        print(f"{str(fit.calib_method):>18} {H:>3} {m['c_index']:>7} "
              f"{m['Nday_risk_brier_skill_raw']:>9} {m['Nday_risk_brier_skill_calibrated']:>9}")
